In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()
ws = wb.active
ws.title = "Table_7"

# Title
ws["A1"] = "Table 7"
ws["A1"].font = Font(bold=True, size=14)
ws["A2"] = "Detailed comparison of the proposed platform and traditional inventory management methods"
ws["A2"].font = Font(italic=True, size=11)
ws.merge_cells("A2:C2")

# Headers
headers = ["Criterion", "Traditional methods", "Proposed platform"]
header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
thin = Side(style="thin", color="BFBFBF")

for col, header in enumerate(headers, start=1):
    cell = ws.cell(row=4, column=col, value=header)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Data
rows = [
    ["Inventory precision", "82.5%", "91%"],
    ["Operational efficiency", "120 min (average inventory counting time)", "65 min (average inventory counting time)"],
    ["Error reduction", "High error rate in manual counting", "40% reduction in overcounting and undercounting errors"],
    ["Customer satisfaction", "Variable product availability and delivery times", "15% improvement in product availability and 20% improvement in delivery punctuality"],
]

for r, row in enumerate(rows, start=5):
    for c, value in enumerate(row, start=1):
        cell = ws.cell(row=r, column=c, value=value)
        cell.alignment = Alignment(vertical="top", wrap_text=True)
        cell.border = Border(bottom=thin)
        if c == 1:
            cell.font = Font(bold=True)

# Formatting
widths = {"A": 26, "B": 40, "C": 42}
for col, width in widths.items():
    ws.column_dimensions[col].width = width

for row in range(4, ws.max_row + 1):
    ws.row_dimensions[row].height = 36

ws.freeze_panes = "A5"

# Add a second simple sheet with plain table only
ws2 = wb.create_sheet("Plain_Table")
for col, header in enumerate(headers, start=1):
    cell = ws2.cell(row=1, column=col, value=header)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

for r, row in enumerate(rows, start=2):
    for c, value in enumerate(row, start=1):
        cell = ws2.cell(row=r, column=c, value=value)
        cell.alignment = Alignment(vertical="top", wrap_text=True)
        cell.border = Border(bottom=thin)
        if c == 1:
            cell.font = Font(bold=True)

for col, width in widths.items():
    ws2.column_dimensions[col].width = width

out = "/mnt/data/table7_inventory_comparison.xlsx"
wb.save(out)
print(f"Saved to {out}")

# Create an LLM Agent Capable of Math and Machine Learning Operations, Using the OpenAI API.

You are an AI engineer working at a financial services firm. Your financial analysts want to use modern large language models as a tool, but need access to mathematical and regression tools that LLM’s are not natively skilled at. Your manager has asked you to prototype an LLM agent solution, coding directly against the OpenAI chat completions API, that augments a chatbot with local tools capable of performing basic mathematical operations and linear regression. Your LLM agent will be evaluated by its ability to successfully compute math and regression problems given as natural language prompts.

In [ ]:
import json
import numpy as np
from sklearn.linear_model import LinearRegression
import openai
from openai import OpenAI

In [ ]:
def perform_math_operation(operation, args=[]):
    result = 0
    if operation == 'add':
        result = sum(args)
    elif operation == 'subtract':
        result = args[0] - sum(args[1:])
    elif operation == 'multiply':
        result = 1
        for num in args:
            result *= num
    elif operation == 'divide':
        result = args[0]
        for num in args[1:]:
            result /= num
    else:
        raise ValueError("Unsupported operation")
 
    retval = {"Result": result}
    return json.dumps(retval)

def predict_next_in_series(series):
    x = np.array(range(len(series))).reshape(-1, 1)
    y = np.array(series).reshape(-1, 1)
    model = LinearRegression().fit(x, y)
    next_index = len(series)
    next_value = model.predict([[next_index]])
    next_info = {
        "Next value": next_value[0][0]
    }
    return json.dumps(next_info)


mathTool = {
        "type": "function",
        "function": {
            "name": "perform_math_operation",
            "description": "Perform a mathematical operation",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": 
                    {
                        "type": "string", 
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "The requested mathematical operation to perform."
                    },
                    "operands": 
                    {
                        "type": "array", 
                        "items": {"type": "number"},
                        "description": "The operands of the requested mathematical operation, from left to right."
                    }
                },
                "required": ["operation", "operands"]
            },
        }
    }

regressionTool = {
        "type": "function",
        "function": {
            "name": "predict_next_in_series",
            "description": "Predict the next number in a series using linear regression",
            "parameters": {
                "type": "object",
                "properties": {
                    "series": {
                        "type": "array", 
                        "items": {"type": "number"},
                        "description": "The list of numbers in the series to be completed."
                    }
                },
                "required": ["series"]
            },
        }
    }

In [ ]:
perform_math_operation("multiply", [8,2,2])

predict_next_in_series([2, 4, 6, 8])

tools = [mathTool, regressionTool]

In [ ]:
client = OpenAI()

def make_initial_request(messages):
 
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        tools=tools,
        tool_choice=None
    )
    
    response_message = response.choices[0].message
    
    print(response_message)
    
    return response_message

def call_tool(tool):
    
    print("calling tool: ")
    print(tool)
    
    response = {}
    
    function_name = tool.function.name
    if (function_name == "perform_math_operation"):
        args = json.loads(tool.function.arguments)
        fn_response = perform_math_operation(operation=args.get("operation"),args=args.get("operands"))
        response = {
            "tool_call_id": tool.id,
            "role": "tool",
            "name": function_name,
            "content": fn_response,
        }
    elif (function_name == "predict_next_in_series"):
        args = json.loads(tool.function.arguments)
        fn_response = predict_next_in_series(series=args.get("series"))
        response = {
            "tool_call_id": tool.id,
            "role": "tool",
            "name": function_name,
            "content": fn_response,
        }
    return response

def llm_agent(prompt):
    messages = [{"role": "user", "content": prompt}]
    
    first_response = make_initial_request(messages)
    
    tool_calls = first_response.tool_calls
    if (tool_calls):
 
        messages.append(first_response)
        
        for tool in tool_calls:
            tool_response = call_tool(tool)
            messages.append(tool_response)
 
        print("second call:")
        print(messages)
        second_response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages = messages,
        )
            
        return second_response
    else:
        return response

In [ ]:
prompt = "What is the next value in the series 1, 3, 5, 7, 11?"
print (llm_agent(prompt))

prompt = "What is 5 divided by 3?"
print (llm_agent(prompt))